In [1]:
from pathlib import Path

fmidv_root = Path(r"C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\data\fmidv")

print("Exists:", fmidv_root.exists())
print("Is dir:", fmidv_root.is_dir())
print("Path:", fmidv_root)

Exists: True
Is dir: True
Path: C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\data\fmidv


In [2]:
from pathlib import Path

def print_tree(root, max_depth=3, max_entries=15, prefix=""):
    root = Path(root)
    if not root.exists():
        print("Path does not exist")
        return

    def _walk(path, depth, prefix):
        if depth > max_depth:
            return
        items = sorted(list(path.iterdir()), key=lambda x: (x.is_file(), x.name.lower()))
        items = items[:max_entries]

        for item in items:
            print(prefix + ("📁 " if item.is_dir() else "📄 ") + item.name)
            if item.is_dir():
                _walk(item, depth + 1, prefix + "    ")

    print(f"Root: {root}")
    _walk(root, 0, prefix)

print_tree(fmidv_root, max_depth=3, max_entries=20)


Root: C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\data\fmidv
📁 forged_ templates
    📁 alb_id
        📄 00_templates_P16_Z2.png
        📄 00_templates_P16_Z4.png
        📄 00_templates_P16_Z6.png
        📄 00_templates_P32_Z2.png
        📄 00_templates_P32_Z4.png
        📄 00_templates_P32_Z6.png
        📄 00_templates_P64_Z2.png
        📄 01_templates_P16_Z2.png
        📄 01_templates_P16_Z4.png
        📄 01_templates_P16_Z6.png
        📄 01_templates_P32_Z2.png
        📄 01_templates_P32_Z4.png
        📄 01_templates_P32_Z6.png
        📄 01_templates_P64_Z2.png
        📄 02_templates_P16_Z2.png
        📄 02_templates_P16_Z4.png
        📄 02_templates_P16_Z6.png
        📄 02_templates_P32_Z2.png
        📄 02_templates_P32_Z4.png
        📄 02_templates_P32_Z6.png
    📁 aze_passport
        📄 00_templates_P16_Z2.png
        📄 00_templates_P16_Z4.png
        📄 00_templates_P16_Z6.png
        📄 00_templates_P32_Z2.png
        📄 00_templates_P32_Z4.png
        📄 00_temp

In [4]:
from pathlib import Path
import pandas as pd
import re
from PIL import Image

fmidv_root = Path(r"C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\data\fmidv")

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

def safe_image_info(path):
    try:
        with Image.open(path) as img:
            return img.size[0], img.size[1], img.mode, None
    except Exception as e:
        return None, None, None, str(e)

rows = []

for path in fmidv_root.rglob("*"):
    if not path.is_file():
        continue
    if path.suffix.lower() not in IMAGE_EXTS:
        continue

    rel = path.relative_to(fmidv_root)
    parts = rel.parts

    # expected structure:
    # top_folder / doc_type / filename
    if len(parts) < 3:
        continue

    top_folder = parts[0].strip()
    doc_type = parts[1].strip()
    filename = path.stem

    # parse filename like:
    # 00_templates_P16_Z2
    # 01_photo_P32_Z6
    # 02_scan_rotated_P64_Z2
    m = re.match(r"(?P<sample_id>\d+?)_(?P<capture_type>.+?)_(?P<patch_size>P\d+)_(?P<zone_count>Z\d+)$", filename)

    sample_id = None
    capture_type = None
    patch_size = None
    zone_count = None

    if m:
        sample_id = m.group("sample_id")
        capture_type = m.group("capture_type")
        patch_size = m.group("patch_size")
        zone_count = m.group("zone_count")

    width, height, mode, read_error = safe_image_info(path)

    rows.append({
        "path": str(path),
        "relative_path": str(rel).replace("\\", "/"),
        "top_folder": top_folder,
        "doc_type": doc_type,
        "filename": path.name,
        "sample_id": sample_id,
        "capture_type": capture_type,
        "patch_size": patch_size,
        "zone_count": zone_count,
        "label": "forged",
        "group_key": f"{doc_type}__{sample_id}" if sample_id is not None else None,
        "width": width,
        "height": height,
        "mode": mode,
        "read_error": read_error,
        "file_size_mb": round(path.stat().st_size / (1024 * 1024), 4),
    })

df_fmidv = pd.DataFrame(rows)

print("Total images:", len(df_fmidv))
print("\nTop folders:")
print(df_fmidv["top_folder"].value_counts())

print("\nDocument types:")
print(df_fmidv["doc_type"].value_counts())

print("\nCapture types:")
print(df_fmidv["capture_type"].value_counts())

print("\nPatch sizes:")
print(df_fmidv["patch_size"].value_counts())

print("\nZone counts:")
print(df_fmidv["zone_count"].value_counts())

print("\nUnreadable files:", df_fmidv["read_error"].notna().sum())

print("\nMost common resolutions:")
print(
    df_fmidv.groupby(["width", "height"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)

print("\nFirst rows:")
print(df_fmidv.head())

Total images: 28000

Top folders:
top_folder
forged_scan_upright    7000
forged_scan_rotated    7000
forged_photo           7000
forged_ templates      7000
Name: count, dtype: int64

Document types:
doc_type
alb_id                  2800
aze_passport            2800
esp_id                  2800
est_id                  2800
fin_id                  2800
grc_passport            2800
lva_passport            2800
rus_internalpassport    2800
srb_passport            2800
svk_id                  2800
Name: count, dtype: int64

Capture types:
capture_type
scan_upright    7000
scan_rotated    7000
photo           7000
templates       7000
Name: count, dtype: int64

Patch sizes:
patch_size
P16    12000
P32    12000
P64     4000
Name: count, dtype: int64

Zone counts:
zone_count
Z2    12000
Z4     8000
Z6     8000
Name: count, dtype: int64

Unreadable files: 0

Most common resolutions:
   width  height  count
0    512     512  28000

First rows:
                                                pat

In [5]:
print("Unique group keys:", df_fmidv["group_key"].nunique())
print(df_fmidv["group_key"].drop_duplicates().sort_values().head(20))


Unique group keys: 1000
0      alb_id__00
7      alb_id__01
14     alb_id__02
21     alb_id__03
28     alb_id__04
35     alb_id__05
42     alb_id__06
49     alb_id__07
56     alb_id__08
63     alb_id__09
70     alb_id__10
77     alb_id__11
84     alb_id__12
91     alb_id__13
98     alb_id__14
105    alb_id__15
112    alb_id__16
119    alb_id__17
126    alb_id__18
133    alb_id__19
Name: group_key, dtype: object


In [7]:
from pathlib import Path
import pandas as pd

RANDOM_STATE = 42
GROUPS_PER_DOC_TYPE = 72
REQUIRED_CAPTURES = {"templates", "photo", "scan_rotated", "scan_upright"}
VARIANT_ORDER = ["P16_Z2", "P16_Z4", "P16_Z6", "P32_Z2", "P32_Z4", "P32_Z6", "P64_Z2"]

df = df_fmidv.copy()
df = df.dropna(subset=["group_key", "doc_type", "capture_type", "patch_size", "zone_count"]).copy()

df["variant"] = df["patch_size"] + "_" + df["zone_count"]

# 1. keep only groups that contain all 4 capture types
group_capture_summary = (
    df.groupby(["group_key", "doc_type"])["capture_type"]
    .agg(lambda x: set(x))
    .reset_index(name="capture_set")
)

valid_groups = group_capture_summary[
    group_capture_summary["capture_set"].apply(lambda x: REQUIRED_CAPTURES.issubset(x))
].copy()

# 2. sample 72 groups per doc_type
sampled_groups = (
    valid_groups.groupby("doc_type", group_keys=False)
    .sample(n=GROUPS_PER_DOC_TYPE, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

selected_group_keys = set(sampled_groups["group_key"])

df_selected = df[df["group_key"].isin(selected_group_keys)].copy()

# 3. assign variants in round-robin within each capture type
selected_rows = []

for capture in sorted(REQUIRED_CAPTURES):
    capture_df = df_selected[df_selected["capture_type"] == capture].copy()

    group_list = sorted(capture_df["group_key"].unique())

    for i, group_key in enumerate(group_list):
        target_variant = VARIANT_ORDER[i % len(VARIANT_ORDER)]

        group_rows = capture_df[capture_df["group_key"] == group_key]

        chosen = group_rows[group_rows["variant"] == target_variant]

        # fallback just in case
        if chosen.empty:
            chosen = group_rows.iloc[[0]]
        else:
            chosen = chosen.iloc[[0]]

        selected_rows.append(chosen)

df_subset = pd.concat(selected_rows, ignore_index=True)

# 4. add final labels
df_subset["source_dataset"] = "fmidv"
df_subset["image_class"] = "forged"

# 5. sort and save
df_subset = df_subset.sort_values(
    ["doc_type", "group_key", "capture_type"]
).reset_index(drop=True)

output_dir = Path(r"C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\data\processed")
output_dir.mkdir(parents=True, exist_ok=True)

subset_path = output_dir / "fmidv_subset_manifest.csv"
df_subset.to_csv(subset_path, index=False)

# 6. checks
print("Subset images:", len(df_subset))
print("Unique groups:", df_subset["group_key"].nunique())

print("\nImages per doc_type:")
print(df_subset["doc_type"].value_counts().sort_index())

print("\nImages per capture_type:")
print(df_subset["capture_type"].value_counts().sort_index())

print("\nVariants:")
print(df_subset["variant"].value_counts().sort_index())

print("\nPatch sizes:")
print(df_subset["patch_size"].value_counts().sort_index())

print("\nZone counts:")
print(df_subset["zone_count"].value_counts().sort_index())

print("\nSaved to:", subset_path)

df_subset.head(12)

df_subset.head(12)

Subset images: 2880
Unique groups: 720

Images per doc_type:
doc_type
alb_id                  288
aze_passport            288
esp_id                  288
est_id                  288
fin_id                  288
grc_passport            288
lva_passport            288
rus_internalpassport    288
srb_passport            288
svk_id                  288
Name: count, dtype: int64

Images per capture_type:
capture_type
photo           720
scan_rotated    720
scan_upright    720
templates       720
Name: count, dtype: int64

Variants:
variant
P16_Z2    412
P16_Z4    412
P16_Z6    412
P32_Z2    412
P32_Z4    412
P32_Z6    412
P64_Z2    408
Name: count, dtype: int64

Patch sizes:
patch_size
P16    1236
P32    1236
P64     408
Name: count, dtype: int64

Zone counts:
zone_count
Z2    1232
Z4     824
Z6     824
Name: count, dtype: int64

Saved to: C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\data\processed\fmidv_subset_manifest.csv


,path,relative_path,top_folder,doc_type,filename,sample_id,capture_type,patch_size,zone_count,label,group_key,width,height,mode,read_error,file_size_mb,variant,source_dataset,image_class
0,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged_photo/alb_id/00_photo_P16_Z2.png,forged_photo,alb_id,00_photo_P16_Z2.png,00,photo,P16,Z2,forged,alb_id__00,512,512,L,None,0.1271,P16_Z2,fmidv,forged
1,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged_scan_rotated/alb_id/00_scan_rotated_P16...,forged_scan_rotated,alb_id,00_scan_rotated_P16_Z2.png,00,scan_rotated,P16,Z2,forged,alb_id__00,512,512,L,None,0.1770,P16_Z2,fmidv,forged
2,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged_scan_upright/alb_id/00_scan_upright_P16...,forged_scan_upright,alb_id,00_scan_upright_P16_Z2.png,00,scan_upright,P16,Z2,forged,alb_id__00,512,512,L,None,0.1908,P16_Z2,fmidv,forged
3,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged_ templates/alb_id/00_templates_P16_Z2.png,forged_ templates,alb_id,00_templates_P16_Z2.png,00,templates,P16,Z2,forged,alb_id__00,512,512,L,None,0.2129,P16_Z2,fmidv,forged
4,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged_photo/alb_id/03_photo_P16_Z4.png,forged_photo,alb_id,03_photo_P16_Z4.png,03,photo,P16,Z4,forged,alb_id__03,512,512,L,None,0.1371,P16_Z4,fmidv,forged
5,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged_scan_rotated/alb_id/03_scan_rotated_P16...,forged_scan_rotated,alb_id,03_scan_rotated_P16_Z4.png,03,scan_rotated,P16,Z4,forged,alb_id__03,512,512,L,None,0.1792,P16_Z4,fmidv,forged
6,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged_scan_upright/alb_id/03_scan_upright_P16...,forged_scan_upright,alb_id,03_scan_upright_P16_Z4.png,03,scan_upright,P16,Z4,forged,alb_id__03,512,512,L,None,0.1925,P16_Z4,fmidv,forged
7,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged_ templates/alb_id/03_templates_P16_Z4.png,forged_ templates,alb_id,03_templates_P16_Z4.png,03,templates,P16,Z4,forged,alb_id__03,512,512,L,None,0.2131,P16_Z4,fmidv,forged
8,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged_photo/alb_id/04_photo_P16_Z6.png,forged_photo,alb_id,04_photo_P16_Z6.png,04,photo,P16,Z6,forged,alb_id__04,512,512,L,None,0.1430,P16_Z6,fmidv,forged
9,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged_scan_rotated/alb_id/04_scan_rotated_P16...,forged_scan_rotated,alb_id,04_scan_rotated_P16_Z6.png,04,scan_rotated,P16,Z6,forged,alb_id__04,512,512,L,None,0.1719,P16_Z6,fmidv,forged


In [4]:
from pathlib import Path
import sys

project_root = Path(r"C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis")
sys.path.append(str(project_root))

print(project_root)
print((project_root / "src").exists())

C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis
True


In [9]:
from pathlib import Path
from src.fmidv_preprocessing import build_fmidv_dataframe, build_fmidv_subset, save_metadata

fmidv_root = Path(r"C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\data\fmidv")

print("FMIDV path:", fmidv_root)
print("Exists:", fmidv_root.exists())

df_fmidv_full = build_fmidv_dataframe(fmidv_root)
print("FULL:", df_fmidv_full.shape)
print(df_fmidv_full["image_class"].value_counts(dropna=False))
print(df_fmidv_full["doc_type"].value_counts(dropna=False))
print(df_fmidv_full["source_type"].value_counts(dropna=False))

df_fmidv_subset = build_fmidv_subset(df_fmidv_full, groups_per_doc_type=72)
print("\nSUBSET:", df_fmidv.shape)
print(df_fmidv["image_class"].value_counts(dropna=False))
print(df_fmidv["doc_type"].value_counts(dropna=False))
print(df_fmidv["source_type"].value_counts(dropna=False))
print(df_fmidv["variant"].value_counts(dropna=False).sort_index())

save_metadata(
    df_fmidv,
    r"C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\data\processed\fmidv_image_manifest.csv"
)

FMIDV path: C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\data\fmidv
Exists: True
FULL: (28000, 17)
image_class
forged    28000
Name: count, dtype: int64
doc_type
alb_id                  2800
aze_passport            2800
esp_id                  2800
est_id                  2800
fin_id                  2800
grc_passport            2800
lva_passport            2800
rus_internalpassport    2800
srb_passport            2800
svk_id                  2800
Name: count, dtype: int64
source_type
templates       7000
photo           7000
scan_rotated    7000
scan_upright    7000
Name: count, dtype: int64

SUBSET: (2880, 17)
image_class
forged    2880
Name: count, dtype: int64
doc_type
alb_id                  288
aze_passport            288
esp_id                  288
est_id                  288
fin_id                  288
grc_passport            288
lva_passport            288
rus_internalpassport    288
srb_passport            288
svk_id                  288
Name: count, dtype:

In [10]:
import pandas as pd

df_check = pd.read_csv(
    r"C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\data\processed\fmidv_image_manifest.csv"
)

print(df_check.shape)
print(df_check["source_type"].value_counts())
print(df_check["doc_type"].value_counts())


(2880, 17)
source_type
photo           720
scan_rotated    720
scan_upright    720
templates       720
Name: count, dtype: int64
doc_type
alb_id                  288
aze_passport            288
esp_id                  288
est_id                  288
fin_id                  288
grc_passport            288
lva_passport            288
rus_internalpassport    288
srb_passport            288
svk_id                  288
Name: count, dtype: int64


In [11]:
df_fmidv_subset.head(12)

,image_path,source_dataset,image_class,group_key,doc_type,source_type,file_name,split_source,original_label,relative_path,top_folder,sample_id,patch_size,zone_count,variant,width,height
0,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,fmidv,forged,alb_id__00,alb_id,photo,00_photo_P16_Z2.png,None,forged,forged_photo\alb_id\00_photo_P16_Z2.png,forged_photo,00,P16,Z2,P16_Z2,512,512
1,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,fmidv,forged,alb_id__00,alb_id,scan_rotated,00_scan_rotated_P16_Z2.png,None,forged,forged_scan_rotated\alb_id\00_scan_rotated_P16...,forged_scan_rotated,00,P16,Z2,P16_Z2,512,512
2,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,fmidv,forged,alb_id__00,alb_id,scan_upright,00_scan_upright_P16_Z2.png,None,forged,forged_scan_upright\alb_id\00_scan_upright_P16...,forged_scan_upright,00,P16,Z2,P16_Z2,512,512
3,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,fmidv,forged,alb_id__00,alb_id,templates,00_templates_P16_Z2.png,None,forged,forged_ templates\alb_id\00_templates_P16_Z2.png,forged_ templates,00,P16,Z2,P16_Z2,512,512
4,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,fmidv,forged,alb_id__03,alb_id,photo,03_photo_P16_Z4.png,None,forged,forged_photo\alb_id\03_photo_P16_Z4.png,forged_photo,03,P16,Z4,P16_Z4,512,512
5,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,fmidv,forged,alb_id__03,alb_id,scan_rotated,03_scan_rotated_P16_Z4.png,None,forged,forged_scan_rotated\alb_id\03_scan_rotated_P16...,forged_scan_rotated,03,P16,Z4,P16_Z4,512,512
6,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,fmidv,forged,alb_id__03,alb_id,scan_upright,03_scan_upright_P16_Z4.png,None,forged,forged_scan_upright\alb_id\03_scan_upright_P16...,forged_scan_upright,03,P16,Z4,P16_Z4,512,512
7,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,fmidv,forged,alb_id__03,alb_id,templates,03_templates_P16_Z4.png,None,forged,forged_ templates\alb_id\03_templates_P16_Z4.png,forged_ templates,03,P16,Z4,P16_Z4,512,512
8,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,fmidv,forged,alb_id__04,alb_id,photo,04_photo_P16_Z6.png,None,forged,forged_photo\alb_id\04_photo_P16_Z6.png,forged_photo,04,P16,Z6,P16_Z6,512,512
9,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,fmidv,forged,alb_id__04,alb_id,scan_rotated,04_scan_rotated_P16_Z6.png,None,forged,forged_scan_rotated\alb_id\04_scan_rotated_P16...,forged_scan_rotated,04,P16,Z6,P16_Z6,512,512


In [1]:
fmidv_sampled = fmidv_df.groupby("source_type", group_keys=False).sample(n=720, random_state=42)


NameError: name 'fmidv_df' is not defined